# Проверка соответствия товарного знака наименованию товара

**Задача #28:** проверить критерий  
> «Значение атрибута „Товарный знак“ не соответствует наименованию товара».

Это финальная PR-версия алгоритмического baseline.

### Что делает ноутбук

1. загружает **весь** `perfumery_brands.csv`;
2. строит доверенный справочник брендов и статистические ассоциации;
3. применяет `check_trademark()` к **каждой записи датасета**;
4. выводит прогресс и скорость обработки;
5. присваивает один из статусов:
   - `MATCH` — соответствие;
   - `LIKELY_MATCH` — вероятное соответствие;
   - `MISMATCH` — несоответствие;
   - `REVIEW` — ручная проверка;
6. сохраняет полный результат в `perfumery_brands_checked_full.csv`;
7. строит сводную статистику по всему датасету.

> Сам CSV в Git коммитить не нужно. Ноутбук умеет брать путь из переменной окружения `PERFUMERY_BRANDS_PATH` или искать файл рядом с проектом.


## Как запускать

Запустите ноутбук **сверху вниз (`Run All`)**.

Необходимые библиотеки:

```text
pandas
rapidfuzz
scikit-learn
```

Датасет должен называться `perfumery_brands.csv`.

Ноутбук ищет его:

1. по пути из переменной окружения `PERFUMERY_BRANDS_PATH`;
2. в текущей папке;
3. в нескольких родительских папках относительно текущей рабочей директории.

После полного прогона создаётся файл:

`perfumery_brands_checked_full.csv`

Он является результатом анализа и **не предназначен для коммита в Git**.


## Словарь основных терминов

| Термин | Русское пояснение |
|---|---|
| `MATCH` | соответствие товарного знака наименованию |
| `LIKELY_MATCH` | вероятное соответствие |
| `MISMATCH` | несоответствие / противоречие |
| `REVIEW` | требуется ручная проверка |
| `UNCERTAIN` | неопределённый случай |
| `Baseline` | базовое решение |
| `Exact match` | точное совпадение |
| `Fuzzy matching` | нечёткое / приблизительное сравнение строк |
| `Trusted brand` | доверенный бренд |
| `Association` | статистическая связь / ассоциация |
| `Coverage` | покрытие автоматическими решениями |
| `Accuracy` | доля правильных автоматических решений |
| `Precision` | точность срабатываний класса |
| `Recall` | полнота |
| `F1-score` | F1-мера |
| `Support` | число примеров класса |
| `Train/test split` | разделение на обучающую и тестовую выборки |
| `Data leakage` | утечка данных |
| `Inference` | применение готового алгоритма к новым данным |
| `Production` | рабочая эксплуатация системы |

> В ML термины `Precision`, `Recall`, `F1-score` обычно сохраняют на английском, поэтому рядом указано русское значение.

In [ ]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from rapidfuzz import fuzz
from sklearn.metrics import precision_recall_fscore_support


In [ ]:
# Русские подписи только для отображения.
# Технические значения статусов и методов алгоритма не меняются.

STATUS_RU = {
    "MATCH": "MATCH (соответствие)",
    "LIKELY_MATCH": "LIKELY_MATCH (вероятное соответствие)",
    "MISMATCH": "MISMATCH (несоответствие)",
    "REVIEW": "REVIEW (ручная проверка)",
    "UNCERTAIN": "UNCERTAIN (неопределённый случай)",
}

METHOD_RU = {
    "missing_data": "отсутствуют необходимые данные",
    "input_quality_review": "проверка качества входного текста",
    "no_brand_conflict_review": "бренд не указан, но в тексте найден бренд",
    "no_brand_review": "бренд не указан — ручная проверка",
    "invalid_brand_format_review": "некорректный формат товарного знака",
    "exact_match": "точное совпадение",
    "verified_relation_match": "проверенная связь аромат/линейка → бренд",
    "verified_relation_conflict": "проверенная связь указывает на другой бренд",
    "association_match": "статистическая ассоциация подтверждает бренд",
    "direct_brand_conflict": "прямое упоминание другого бренда",
    "fuzzy_match": "нечёткое совпадение",
    "trusted_brand_conflict": "конфликт с доверенным брендом",
    "insufficient_evidence": "недостаточно данных для автоматического решения",
}

def status_ru(status):
    return STATUS_RU.get(status, status)

def method_ru(method):
    return METHOD_RU.get(method, method)


In [ ]:
# Автоматический поиск датасета.
# При необходимости путь можно явно передать через:
# PERFUMERY_BRANDS_PATH=/path/to/perfumery_brands.csv

env_path = os.getenv("PERFUMERY_BRANDS_PATH")

candidate_paths = []

if env_path:
    candidate_paths.append(Path(env_path))

cwd = Path.cwd()

candidate_paths.extend([
    cwd / "perfumery_brands.csv",
    cwd.parent / "perfumery_brands.csv",
    cwd.parent.parent / "perfumery_brands.csv",
    cwd.parent.parent.parent / "perfumery_brands.csv",
])

DATA_PATH = next(
    (path for path in candidate_paths if path.exists()),
    None
)

if DATA_PATH is None:
    checked = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        "Не найден perfumery_brands.csv.\n\n"
        "Проверенные пути:\n"
        f"{checked}\n\n"
        "Можно указать путь через переменную окружения "
        "PERFUMERY_BRANDS_PATH."
    )

REQUIRED_COLUMNS = ["brand", "brand_in_text", "text"]

print(f"Используется датасет: {DATA_PATH.resolve()}")


## 1. Загрузка данных

Для работы нужен `perfumery_brands.csv` в той же папке.

Используются поля:

- `brand` — заявленный товарный знак;
- `brand_in_text` — бренд, найденный/указанный в исходных данных;
- `text` — наименование товара.

`brand_in_text` используется при построении статистических справочников, но не является обязательным входом `inference`-функции (**функции применения готового алгоритма**).

In [ ]:
data = pd.read_csv(
    DATA_PATH,
    usecols=REQUIRED_COLUMNS
)

missing_columns = set(REQUIRED_COLUMNS) - set(data.columns)

if missing_columns:
    raise ValueError(
        f"В датасете отсутствуют обязательные столбцы: "
        f"{sorted(missing_columns)}"
    )

print(f"Размер датасета: {data.shape[0]:,} строк × {data.shape[1]} столбца")
display(data.head())


## 2. Краткий анализ данных

Проверяются пропуски и количество уникальных значений перед построением справочников.

Это диагностический этап: он помогает понять качество и структуру исходного датасета.

In [ ]:
data[REQUIRED_COLUMNS].isna().sum()

In [ ]:
data["brand"].nunique(), data["brand_in_text"].nunique()

## Нормализация текста

`Normalization` (**нормализация**) приводит строки к единому виду:

- нижний регистр;
- `ё → е`;
- удаление пунктуации;
- удаление лишних пробелов.

Дополнительная нормализация обрабатывает `mixed-script` (**смешение кириллицы и латиницы**) внутри одного слова, например визуально похожие `р/p`, `а/a`, `м/m`.

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
confusable_map = str.maketrans({
    "а": "a",
    "е": "e",
    "о": "o",
    "р": "p",
    "с": "c",
    "у": "y",
    "х": "x",
    "к": "k",
    "м": "m",
    "т": "t",
    "в": "b",
    "н": "h",
})


def normalize_for_matching(text):
    text = normalize_text(text)

    if text is None:
        return None

    result = []

    for word in text.split():

        has_latin = bool(
            re.search(r"[a-z]", word)
        )

        has_cyrillic = bool(
            re.search(r"[а-яё]", word)
        )

        # Исправляем только слова,
        # где одновременно есть латиница и кириллица
        if has_latin and has_cyrillic:
            word = word.translate(confusable_map)

        result.append(word)

    return " ".join(result)

## Служебные правила и проверенные связи

Здесь задаются:

- специальные значения отсутствующего товарного знака;
- подтверждённые `aliases` (**варианты написания брендов**);
- вручную проверенные связи `аромат/линейка → бренд`.

Проверенные связи имеют приоритет над шумными статистическими ассоциациями.

In [ ]:
no_brand_values = {
    "без товарного знака",
    "безтоварного знака",
    "без бренда",
    "бренд отсутствует",
    "товарный знак отсутствует",
    "отсутствует",
    "нет",
    "без знака",
    "без товарной марки",
    "не указан",
}

brand_aliases = {
    "byredo parfums": "byredo",
    "francis kurkdjian": "maison francis kurkdjian",
    "parfum francis kurkdjian": "maison francis kurkdjian",
}

verified_product_relations = {
    "domenico gusto": "christine lavoisier parfums",
    "oudi": "al haramain",
    "russian police clean hands": "sergio nero",
    "the only one": "dolce gabbana",
    "stronger with you intensely": "giorgio armani",
    "festival splashes": "mexx",
    "ma cherry": "adopt",
    "hyrax": "zoologist",
}

## 3. Доверенный справочник брендов

Из исторических пар `brand_in_text → brand` строится `trusted brand catalog` (**доверенный справочник брендов**).

В него попадают только устойчивые значения с высокой долей доминирующего бренда (`dominance ≥ 0.95`).

Цель — использовать только достаточно надёжные брендовые сигналы и снизить риск ложного `MISMATCH`.

In [ ]:
brand_data = data[["brand", "brand_in_text"]].copy()

brand_data["brand_normalized"] = brand_data["brand"].apply(normalize_text)
brand_data["brand_in_text_normalized"] = brand_data["brand_in_text"].apply(normalize_text)

clean_pairs_data = brand_data.dropna(
    subset=["brand_normalized", "brand_in_text_normalized"]
).copy()

clean_pairs_data = clean_pairs_data[
    (~clean_pairs_data["brand_normalized"].isin(no_brand_values))
    & (~clean_pairs_data["brand_normalized"].str.isnumeric())
]

clean_pair_counts = (
    clean_pairs_data
    .groupby(["brand_in_text_normalized", "brand_normalized"])
    .size()
    .reset_index(name="count")
)

# Доверенный справочник прямых брендов
clean_dominant_brand = (
    clean_pair_counts
    .sort_values(["brand_in_text_normalized", "count"], ascending=[True, False])
    .drop_duplicates("brand_in_text_normalized")
    .rename(columns={"brand_normalized": "dominant_brand", "count": "dominant_count"})
)

clean_candidate_totals = (
    clean_pair_counts
    .groupby("brand_in_text_normalized")["count"]
    .sum()
    .reset_index(name="total")
)

clean_dominant_stats = clean_dominant_brand.merge(
    clean_candidate_totals,
    on="brand_in_text_normalized"
)
clean_dominant_stats["dominance"] = (
    clean_dominant_stats["dominant_count"] / clean_dominant_stats["total"]
)

trusted_brands = clean_dominant_stats[
    (clean_dominant_stats["total"] >= 20)
    & (clean_dominant_stats["dominance"] >= 0.95)
    & (
        clean_dominant_stats["brand_in_text_normalized"]
        == clean_dominant_stats["dominant_brand"]
    )
].copy()

trusted_brand_catalog = trusted_brands["brand_in_text_normalized"].tolist()

print("Доверенных брендов:", len(trusted_brand_catalog))

In [ ]:
# Быстрый поиск доверенных брендов.
# Вместо отдельного поиска каждого бренда строится одно регулярное выражение.

trusted_brand_terms = sorted(
    [
        brand
        for brand in trusted_brand_catalog
        if len(brand) >= 4
    ],
    key=len,
    reverse=True
)

TRUSTED_BRAND_PATTERN = (
    re.compile(
        r"(?<!\w)(?:"
        + "|".join(re.escape(term) for term in trusted_brand_terms)
        + r")(?!\w)"
    )
    if trusted_brand_terms
    else None
)


def find_trusted_brands(text):
    normalized_text = normalize_text(text)

    if normalized_text is None or TRUSTED_BRAND_PATTERN is None:
        return []

    found = {
        match.group(0)
        for match in TRUSTED_BRAND_PATTERN.finditer(normalized_text)
    }

    return sorted(found, key=len, reverse=True)


In [ ]:
def brands_equivalent(brand_1, brand_2):
    brand_1 = normalize_text(brand_1)
    brand_2 = normalize_text(brand_2)

    if brand_1 is None or brand_2 is None:
        return False

    # Полное совпадение
    if brand_1 == brand_2:
        return True

    # Один вариант входит в другой
    if brand_1 in brand_2 or brand_2 in brand_1:
        return True

    # Небольшая разница в написании
    similarity = fuzz.ratio(brand_1, brand_2)

    if similarity >= 90:
        return True

    return False

In [ ]:
def detect_trusted_mismatch(brand, text):
    detected_brands = find_trusted_brands(text)

    conflicts = []

    for detected_brand in detected_brands:
        if not brands_equivalent(brand, detected_brand):
            conflicts.append(detected_brand)

    if conflicts:
        return "MISMATCH", conflicts

    return "NO_CONFLICT_FOUND", []

## 4. Канонизация и устойчивые статистические ассоциации

`Canonicalization` (**канонизация**) объединяет разные варианты одного бренда.

После этого строятся статистические связи между фрагментом наименования и брендом.

Текущие условия `baseline` (**базового решения**):

- `top1_count ≥ 20`;
- `top_ratio ≥ 3`;
- `count_gap ≥ 20`;
- длина термина ≥ 4.

Более длинные и конкретные совпадения имеют приоритет над вложенными короткими значениями.

In [ ]:
def canonicalize_brand(brand):
    brand = normalize_text(brand)

    if brand is None:
        return None

    return brand_aliases.get(brand, brand)

In [ ]:
canonical_pairs = clean_pair_counts.copy()
canonical_pairs["canonical_brand"] = canonical_pairs["brand_normalized"].apply(canonicalize_brand)

canonical_pair_counts = (
    canonical_pairs
    .groupby(["brand_in_text_normalized", "canonical_brand"], as_index=False)["count"]
    .sum()
)

canonical_ranked = canonical_pair_counts.copy()
canonical_ranked["rank"] = (
    canonical_ranked
    .groupby("brand_in_text_normalized")["count"]
    .rank(method="first", ascending=False)
)

canonical_top1 = (
    canonical_ranked[canonical_ranked["rank"] == 1]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top1_brand", "count": "top1_count"})
)

canonical_top2 = (
    canonical_ranked[canonical_ranked["rank"] == 2]
    [["brand_in_text_normalized", "canonical_brand", "count"]]
    .rename(columns={"canonical_brand": "top2_brand", "count": "top2_count"})
)

canonical_strength = canonical_top1.merge(
    canonical_top2,
    on="brand_in_text_normalized",
    how="left"
)

canonical_strength["top_ratio"] = (
    canonical_strength["top1_count"] / canonical_strength["top2_count"]
)
canonical_strength["count_gap"] = (
    canonical_strength["top1_count"] - canonical_strength["top2_count"]
)

strong_associations = canonical_strength[
    (canonical_strength["top1_count"] >= 20)
    & (canonical_strength["top_ratio"] >= 3)
    & (canonical_strength["count_gap"] >= 20)
    & (canonical_strength["brand_in_text_normalized"].str.len() >= 4)
].copy()

association_map = dict(
    zip(
        strong_associations["brand_in_text_normalized"],
        strong_associations["top1_brand"]
    )
)

print("Устойчивых статистических связей:", len(association_map))

In [ ]:
# Быстрый поиск статистических ассоциаций.
# Старый вариант проходил по всем association_map для каждого товара.
# Здесь все термины объединяются в одно скомпилированное выражение.

association_term_to_brand = {}

for term, associated_brand in association_map.items():
    normalized_term = normalize_for_matching(term)

    if normalized_term is not None:
        association_term_to_brand[normalized_term] = associated_brand

association_terms = sorted(
    association_term_to_brand,
    key=len,
    reverse=True
)

ASSOCIATION_PATTERN = (
    re.compile(
        r"(?<!\w)(?:"
        + "|".join(re.escape(term) for term in association_terms)
        + r")(?!\w)"
    )
    if association_terms
    else None
)


def find_associated_brands(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None or ASSOCIATION_PATTERN is None:
        return []

    # Термины в pattern уже отсортированы по длине.
    # При совпадениях с одной позиции более длинный термин имеет приоритет.
    matches = []

    for match in ASSOCIATION_PATTERN.finditer(normalized_text):
        term = match.group(0)

        matches.append(
            {
                "term": term,
                "brand": association_term_to_brand[term],
            }
        )

    return matches


## 5. Проверенные связи и проверки качества данных

Здесь находятся:

1. `Verified relations` — **проверенные связи аромат/линейка → бренд**.
2. `Input quality checks` — **проверки качества входного текста**.
3. `Brand quality checks` — **проверка формата значения товарного знака**.

Подозрительные входы безопаснее отправлять в `REVIEW` (**ручную проверку**), а не автоматически объявлять `MISMATCH`.

In [ ]:
# Быстрый поиск проверенных связей аромат/линейка → бренд.

verified_term_map = {}

for term, brand in verified_product_relations.items():
    normalized_term = normalize_for_matching(term)

    if normalized_term is not None:
        verified_term_map[normalized_term] = {
            "term": term,
            "brand": brand,
        }

verified_terms = sorted(
    verified_term_map,
    key=len,
    reverse=True
)

VERIFIED_RELATION_PATTERN = (
    re.compile(
        r"(?<!\w)(?:"
        + "|".join(re.escape(term) for term in verified_terms)
        + r")(?!\w)"
    )
    if verified_terms
    else None
)


def find_verified_product_relation(text):
    normalized_text = normalize_for_matching(text)

    if normalized_text is None or VERIFIED_RELATION_PATTERN is None:
        return None

    # Старый алгоритм отдавал приоритет самому длинному найденному термину.
    found_terms = [
        match.group(0)
        for match in VERIFIED_RELATION_PATTERN.finditer(normalized_text)
    ]

    if not found_terms:
        return None

    best_term = max(found_terms, key=len)

    return verified_term_map[best_term].copy()


In [ ]:
def check_input_quality(text):
    if pd.isna(text):
        return {
            "is_valid": False,
            "reason": "Отсутствует текст товара"
        }

    text = str(text)

    text_length = len(text)

    barcode_like_count = len(
        re.findall(
            r"\b\d{12,14}\b",
            text
        )
    )

    if text_length > 500:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": "Аномально длинное наименование товара"
        }

    if barcode_like_count >= 3:
        return {
            "is_valid": False,
            "text_length": text_length,
            "barcode_like_count": barcode_like_count,
            "reason": (
                "В тексте обнаружено несколько числовых "
                "идентификаторов, возможна склейка товаров"
            )
        }

    return {
        "is_valid": True,
        "text_length": text_length,
        "barcode_like_count": barcode_like_count
    }

In [ ]:
def check_brand_input_quality(brand):
    if pd.isna(brand):
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    normalized_brand = normalize_for_matching(brand)

    if normalized_brand is None:
        return {
            "is_valid": False,
            "reason": "Отсутствует значение товарного знака"
        }

    # 1. Значение состоит только из цифр
    if normalized_brand.isdigit():
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака состоит только из цифр "
                "и требует отдельной проверки корректности атрибута"
            )
        }

    # 2. Сомнительные служебные значения текущего датасета
    suspicious_brand_values = {
        "aroma"
    }

    if normalized_brand in suspicious_brand_values:
        return {
            "is_valid": False,
            "reason": (
                "Значение товарного знака является сомнительным "
                "служебным значением в текущем датасете"
            )
        }

    # 3. Значение похоже на наименование юридического лица
    legal_entity_patterns = [
        r"^(ооо|оао|зао|пао|ао)\b",
        r"^ип\b",
        r"^общество с ограниченной ответственностью\b",
        r"^(llc|ltd|inc)\b"
    ]

    for pattern in legal_entity_patterns:
        if re.search(pattern, normalized_brand):
            return {
                "is_valid": False,
                "reason": (
                    "Значение похоже на наименование юридического "
                    "лица или производителя и требует отдельной проверки"
                )
            }

    return {
        "is_valid": True
    }

In [ ]:
def find_direct_brand_conflict(
    brand,
    text=None,
    associations=None
):
    declared_brand = normalize_for_matching(
        canonicalize_brand(brand)
    )

    # Если associations уже были найдены в check_trademark,
    # используем их повторно и не сканируем текст второй раз.
    if associations is None:
        associations = find_associated_brands(text)

    for association in associations:
        term = normalize_for_matching(
            association["term"]
        )

        associated_brand = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Сильный случай:
        # термин сам является названием бренда
        if term == associated_brand:
            if declared_brand != associated_brand:
                return {
                    "brand": associated_brand,
                    "term": association["term"]
                }

    return None


## 6. Финальная функция проверки

`check_trademark(brand, text)` — итоговая `inference`-функция (**применение готового алгоритма к новым данным**).

Порядок принятия решения:

1. отсутствующие данные;
2. качество входного текста;
3. специальные значения отсутствующего бренда;
4. качество значения `brand`;
5. `exact match` — точное совпадение;
6. `verified relations` — проверенные связи;
7. `statistical associations` — статистические ассоциации;
8. `direct brand conflict` — прямой конфликт брендов;
9. `fuzzy matching` — нечёткое сравнение;
10. `trusted brand conflict` — конфликт с доверенным брендом;
11. недостаточно доказательств → `REVIEW` (**ручная проверка**).

In [ ]:
def check_trademark(brand, text):
    # Нормализация
    normalized_brand = normalize_for_matching(brand)
    normalized_text = normalize_for_matching(text)

    # 1. Проверка отсутствующих данных
    if normalized_brand is None or normalized_text is None:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "missing_data",
            "reason": "Отсутствуют необходимые данные"
        }

    # 2. Проверка качества входного текста
    quality_result = check_input_quality(text)

    if not quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "input_quality_review",
            "text_length": quality_result.get("text_length"),
            "barcode_like_count": quality_result.get(
                "barcode_like_count"
            ),
            "reason": quality_result["reason"]
        }

    # 3. Значения:
    # "Без товарного знака", "нет", "отсутствует", "не указан"
    if normalized_brand in no_brand_values:
        detected_brands = find_trusted_brands(text)

        if detected_brands:
            return {
                "status": "REVIEW",
                "declared_brand": brand,
                "detected_brands": detected_brands,
                "method": "no_brand_conflict_review",
                "reason": (
                    "Указано отсутствие товарного знака, "
                    "но в наименовании найдено упоминание бренда"
                )
            }

        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "no_brand_review",
            "reason": (
                "Указано отсутствие товарного знака. "
                "Недостаточно данных для автоматического подтверждения"
            )
        }

    # 4. Проверка формата значения товарного знака
    brand_quality_result = check_brand_input_quality(brand)

    if not brand_quality_result["is_valid"]:
        return {
            "status": "REVIEW",
            "declared_brand": brand,
            "method": "invalid_brand_format_review",
            "reason": brand_quality_result["reason"]
        }

    # 5. Точное вхождение бренда в наименование
    if normalized_brand in normalized_text:
        return {
            "status": "MATCH",
            "declared_brand": brand,
            "similarity": 100,
            "method": "exact_match",
            "reason": "Товарный знак найден в наименовании товара"
        }

    # 6. Проверенные вручную связи:
    # название аромата / линейки -> реальный бренд
    verified_relation = find_verified_product_relation(text)

    if verified_relation is not None:
        declared_canonical = normalize_for_matching(
            canonicalize_brand(brand)
        )

        verified_canonical = normalize_for_matching(
            canonicalize_brand(
                verified_relation["brand"]
            )
        )

        # Связь подтверждает заявленный бренд
        if declared_canonical == verified_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": verified_relation["term"],
                "associated_brand": verified_relation["brand"],
                "method": "verified_relation_match",
                "reason": (
                    "В наименовании найдено название аромата "
                    "или линейки, связанное с заявленным товарным знаком"
                )
            }

        # Связь указывает на другой бренд
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                verified_relation["brand"]
            ],
            "associated_term": verified_relation["term"],
            "method": "verified_relation_conflict",
            "reason": (
                "В наименовании найдено название аромата "
                "или линейки, связанное с другим товарным знаком"
            )
        }

    # 7. Автоматически найденные устойчивые ассоциации
    associations = find_associated_brands(text)

    declared_canonical = normalize_for_matching(
        canonicalize_brand(brand)
    )

    for association in associations:
        associated_canonical = normalize_for_matching(
            canonicalize_brand(
                association["brand"]
            )
        )

        # Ассоциация подтверждает заявленный бренд
        if declared_canonical == associated_canonical:
            return {
                "status": "MATCH",
                "declared_brand": brand,
                "associated_term": association["term"],
                "associated_brand": association["brand"],
                "method": "association_match",
                "reason": (
                    "В наименовании найдено название, "
                    "устойчиво связанное с заявленным товарным знаком"
                )
            }

    # 8. Проверка прямого упоминания другого бренда
    direct_conflict = find_direct_brand_conflict(
        brand,
        associations=associations
    )

    if direct_conflict is not None:
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": [
                direct_conflict["brand"]
            ],
            "associated_term": direct_conflict["term"],
            "method": "direct_brand_conflict",
            "reason": (
                "В наименовании явно обнаружен другой "
                "товарный знак"
            )
        }

    # 9. Fuzzy matching
    similarity = fuzz.partial_ratio(
        normalized_brand,
        normalized_text
    )

    if similarity >= 90:
        return {
            "status": "LIKELY_MATCH",
            "declared_brand": brand,
            "similarity": similarity,
            "method": "fuzzy_match",
            "reason": (
                "Обнаружено высокое сходство "
                "товарного знака и наименования"
            )
        }

    # 10. Поиск другого доверенного бренда
    conflict_status, conflicts = detect_trusted_mismatch(
        brand,
        text
    )

    if conflict_status == "MISMATCH":
        return {
            "status": "MISMATCH",
            "declared_brand": brand,
            "detected_brands": conflicts,
            "similarity": similarity,
            "method": "trusted_brand_conflict",
            "reason": (
                "В наименовании обнаружен другой "
                "доверенный товарный знак"
            )
        }

    # 11. Недостаточно информации
    return {
        "status": "REVIEW",
        "declared_brand": brand,
        "similarity": similarity,
        "method": "insufficient_evidence",
        "reason": (
            "Недостаточно данных для автоматического решения"
        )
    }

### Smoke-тесты финальной функции

`Smoke tests` — короткие контрольные проверки ключевых веток алгоритма:

- точное совпадение;
- совпадение по ассоциации;
- нечёткое совпадение;
- явный конфликт;
- отсутствие товарного знака;
- прямое упоминание другого бренда.

In [ ]:
tests = [
    ("PARFUMS DE MARLY", "Parfums de Marly Delina Exclusif"),
    ("Byredo", "Парфюмерная вода Gypsy Water"),
    ("Franck Oliver", "FRANCK OLIVIER IN PINK"),
    ("Dior", "Police The Legendary Scent"),
    ("НЕ УКАЗАН", "PRADA AMBER POUR HOMME"),
    ("Mr.Tooth", "M.MICALLEF парфюмерная вода ДЕСИРТОКСИК 100ml"),
]

for brand, text in tests:
    result = check_trademark(brand, text)
    print(f"{brand!r} -> {result['status']} / {result.get('method')}")

print("\nРусская расшифровка smoke-тестов:")
for brand, text in tests:
    result = check_trademark(brand, text)
    print(
        f"{brand!r} -> "
        f"{status_ru(result['status'])} / "
        f"{method_ru(result.get('method'))}"
    )


## 7. Полный анализ всего датасета

Здесь **нет `.sample(n=5000)`**: алгоритм применяется ко всем строкам.

Для ускорения:

- доверенные бренды собраны в один предварительно скомпилированный regex;
- статистические ассоциации также ищутся одним regex;
- проверенные связи компилируются заранее;
- найденные ассоциации переиспользуются в `direct_brand_conflict`;
- основной цикл идёт через `zip`, без `DataFrame.apply(axis=1)`.

Перед полным запуском выполняется короткий speed-test на 5000 строках.


### Проверка скорости перед полным запуском

Перед обработкой 1,49 млн строк рекомендуется сначала проверить скорость
на 5000 строках. Это позволяет убедиться, что используется именно ускоренная
версия функций.

In [ ]:
speed_sample = data[
    ["brand", "text"]
].sample(
    n=min(5000, len(data)),
    random_state=42
)

speed_start = time.time()

_ = [
    check_trademark(brand, text)
    for brand, text in zip(
        speed_sample["brand"],
        speed_sample["text"]
    )
]

speed_elapsed = time.time() - speed_start
speed_rows_per_sec = len(speed_sample) / speed_elapsed

estimated_minutes = (
    len(data)
    / speed_rows_per_sec
    / 60
)

print(
    f"Тест скорости: {speed_rows_per_sec:.1f} строк/с "
    f"({speed_elapsed:.2f} сек на {len(speed_sample):,} строк)"
)

print(
    f"Грубая оценка полного прогона: "
    f"{estimated_minutes:.1f} мин"
)


In [ ]:
# Полный датасет — анализ каждой строки

full_df = data[
    ["brand", "text", "brand_in_text"]
].copy()

print(f"Всего строк для анализа: {len(full_df):,}")

results = []
start_time = time.time()
total_rows = len(full_df)

for i, (brand, text) in enumerate(
    zip(full_df["brand"], full_df["text"]),
    start=1
):
    results.append(
        check_trademark(
            brand,
            text
        )
    )

    if i % 10000 == 0 or i == total_rows:
        elapsed = time.time() - start_time
        speed = i / elapsed if elapsed > 0 else 0
        remaining = (total_rows - i) / speed if speed > 0 else 0

        print(
            f"Обработано: {i:,} / {total_rows:,} "
            f"({i / total_rows:.1%}) | "
            f"{speed:.1f} строк/с | "
            f"осталось примерно {remaining / 60:.1f} мин"
        )

full_df["check_result"] = results

result_fields = {
    "final_status": "status",
    "final_method": "method",
    "reason": "reason",
    "similarity": "similarity",
    "detected_brands": "detected_brands",
    "associated_term": "associated_term",
    "associated_brand": "associated_brand",
}

for column, key in result_fields.items():
    full_df[column] = full_df["check_result"].apply(
        lambda x, k=key: x.get(k)
    )

full_df["final_status_ru"] = full_df["final_status"].apply(status_ru)
full_df["final_method_ru"] = full_df["final_method"].apply(method_ru)

elapsed_total = time.time() - start_time

print()
print(f"Полный анализ завершён за {elapsed_total / 60:.2f} мин.")

status_counts = full_df["final_status"].value_counts()

status_table = pd.DataFrame({
    "status": status_counts.index,
    "status_ru": [status_ru(x) for x in status_counts.index],
    "count": status_counts.values,
    "percent": [
        round(count / len(full_df) * 100, 2)
        for count in status_counts.values
    ],
})

print("\nИтоговое распределение статусов:")
display(status_table)

print("\nРаспределение методов:")
method_counts = full_df["final_method"].value_counts()

display(
    pd.DataFrame({
        "method": method_counts.index,
        "method_ru": [method_ru(x) for x in method_counts.index],
        "count": method_counts.values,
    })
)


### Сохранение полного результата

Полный результат сохраняется в CSV.  
В файл попадают исходные поля и объяснение решения алгоритма.

> Этот CSV может быть большим. Его **не нужно добавлять в Pull Request**.


In [ ]:
export_columns = [
    "brand",
    "text",
    "brand_in_text",
    "final_status",
    "final_status_ru",
    "final_method",
    "final_method_ru",
    "reason",
    "similarity",
    "detected_brands",
    "associated_term",
    "associated_brand",
]

full_result_df = full_df[export_columns].copy()

OUTPUT_PATH = Path("perfumery_brands_checked_full.csv")

full_result_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Результат сохранён: {OUTPUT_PATH.resolve()}")
print(f"Сохранено строк: {len(full_result_df):,}")

display(full_result_df.head(20))


### Сводный анализ результата

После полного прогона автоматически формируются:

- доля каждого статуса;
- количество `MISMATCH`;
- распределение причин/методов;
- примеры `MISMATCH`;
- количество `REVIEW`.

Это позволяет получить итоговый ответ по всему датасету без ручного просмотра миллионов строк.


In [ ]:
status_summary = (
    full_df["final_status"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

status_summary["percent"] = (
    status_summary["count"]
    / len(full_df)
    * 100
).round(2)

status_summary["status_ru"] = [
    status_ru(x)
    for x in status_summary.index
]

print("Сводка по статусам:")
display(status_summary)

print("\nСводка по методам:")
method_summary = (
    full_df["final_method"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)

method_summary["percent"] = (
    method_summary["count"]
    / len(full_df)
    * 100
).round(2)

method_summary["method_ru"] = [
    method_ru(x)
    for x in method_summary.index
]

display(method_summary)

full_mismatch_df = full_df[
    full_df["final_status"] == "MISMATCH"
].copy()

full_review_df = full_df[
    full_df["final_status"] == "REVIEW"
].copy()

print(
    f"\nВсего MISMATCH (несоответствий): "
    f"{len(full_mismatch_df):,}"
)

print(
    f"Всего REVIEW (ручная проверка): "
    f"{len(full_review_df):,}"
)

print("\nПервые 100 MISMATCH:")
display(
    full_mismatch_df[
        [
            "brand",
            "text",
            "brand_in_text",
            "final_method_ru",
            "detected_brands",
            "reason",
        ]
    ].head(100)
)


## 8. Ранее проведённая ручная оценка baseline

Этот блок **не влияет на полный анализ датасета**.

Он фиксирует результаты отдельной ручной проверки 100 товаров, выполненной во время разработки алгоритма:

- 68 `MATCH`;
- 19 `MISMATCH`;
- 13 `UNCERTAIN`.

После исключения `UNCERTAIN`:

- Coverage: **62.07%**
- Accuracy среди автоматически принятых решений: **98.15%**
- `MATCH`: Precision **100%**, Recall **73.53%**, F1 **84.75%**
- `MISMATCH`: Precision **75%**, Recall **15.79%**, F1 **26.09%**

Метрики приведены как характеристика текущего baseline и не требуют повторной ручной разметки при каждом полном запуске.


In [ ]:
# Итоговая ручная оценка: 100 товаров, из них 13 UNCERTAIN.
manual_label_distribution = pd.Series({
    "MATCH": 68,
    "MISMATCH": 19,
    "UNCERTAIN": 13,
}, name="count")

final_confusion = pd.DataFrame(
    {
        "MATCH": [50, 0],
        "MISMATCH": [1, 3],
        "REVIEW": [17, 16],
    },
    index=pd.Index(["MATCH", "MISMATCH"], name="manual_label")
)
final_confusion.columns.name = "predicted_label"

# Восстанавливаем y_true/y_pred из подтверждённой матрицы, чтобы пересчитать метрики.
y_true = (
    ["MATCH"] * 68
    + ["MISMATCH"] * 19
)
y_pred = (
    ["MATCH"] * 50
    + ["MISMATCH"] * 1
    + ["REVIEW"] * 17
    + ["MISMATCH"] * 3
    + ["REVIEW"] * 16
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=["MATCH", "MISMATCH"],
    zero_division=0
)

metrics_df = pd.DataFrame({
    "class": ["MATCH", "MISMATCH"],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})

automatic_total = 50 + 1 + 3
automatic_correct = 50 + 3
coverage = automatic_total / 87
automatic_accuracy = automatic_correct / automatic_total

print("Распределение ручных меток:")
print(manual_label_distribution)
print("\nМатрица итоговой оценки:")
display(final_confusion)
print(f"Coverage: {coverage:.2%}")
print(f"Accuracy среди автоматических решений: {automatic_accuracy:.2%}")
display(metrics_df)

print("\nРусскоязычное представление итоговой оценки:")

display(pd.DataFrame({
    "Ручная метка": manual_label_distribution.index,
    "Русское пояснение": [status_ru(x) for x in manual_label_distribution.index],
    "Количество": manual_label_distribution.values,
}))

final_confusion_ru = final_confusion.rename(
    index={
        "MATCH": "MATCH (реальное соответствие)",
        "MISMATCH": "MISMATCH (реальное несоответствие)",
    },
    columns={
        "MATCH": "MATCH (алгоритм: соответствие)",
        "MISMATCH": "MISMATCH (алгоритм: несоответствие)",
        "REVIEW": "REVIEW (алгоритм: ручная проверка)",
    },
)

display(final_confusion_ru)

metrics_df_ru = metrics_df.rename(columns={
    "class": "Класс",
    "precision": "Precision (точность срабатываний)",
    "recall": "Recall (полнота)",
    "f1": "F1-score (F1-мера)",
    "support": "Support (число примеров)",
})

metrics_df_ru["Класс"] = metrics_df_ru["Класс"].map({
    "MATCH": "MATCH (соответствие)",
    "MISMATCH": "MISMATCH (несоответствие)",
})

print(f"Coverage (покрытие): {coverage:.2%}")
print(
    "Accuracy (доля правильных автоматических решений): "
    f"{automatic_accuracy:.2%}"
)
display(metrics_df_ru)


## 9. Что считать результатом

После `Run All` основными результатами являются:

1. `status_table` — количество и доля `MATCH / MISMATCH / REVIEW / LIKELY_MATCH`;
2. `method_summary` — причины/методы решений;
3. `full_mismatch_df` — все найденные автоматические несоответствия;
4. `full_review_df` — записи, которым нужна дополнительная проверка;
5. `perfumery_brands_checked_full.csv` — полный результат по каждой строке.

Именно эти данные используются для общего вывода по всему датасету.


## 10. Следующий этап

После получения полного результата:

1. проверить репрезентативную выборку `MISMATCH`;
2. проверить выборку `REVIEW`;
3. уточнить качество алгоритма на этих группах;
4. использовать сложные `REVIEW`-случаи для следующего ML-этапа.

Алгоритмический baseline остаётся первым безопасным уровнем проверки.


## Справочные материалы

- RapidFuzz `partial_ratio` — нечёткое сравнение строк: https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html
- scikit-learn Precision/Recall — точность срабатываний и полнота: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- scikit-learn `classification_report` — Precision / Recall / F1: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
- Jupyter Notebook — код, Markdown и результаты в одном документе: https://jupyter-notebook.readthedocs.io/en/latest/notebook.html

## Для Pull Request

Рекомендуемое расположение:

`model/notebooks/trademark_consistency_full_dataset.ipynb`

В PR добавляется **только ноутбук**.

Не добавлять:

- `perfumery_brands.csv`;
- `perfumery_brands_checked_full.csv`;
- `.ipynb_checkpoints/`.

Датасет имеет большой размер и должен передаваться отдельно от Git-репозитория.
